# 01 — Exploration et inventaire de la base SQLite

Objectif de cette première étape : comprendre précisément la structure de la base, identifier les tables, les colonnes, les volumes, les statuts, les doublons, les clés de jointure et les premières anomalies visibles.

Cette étape ne nettoie pas encore les données. Elle sert uniquement à établir un inventaire fiable et documenté avant le nettoyage.

In [ ]:
# Import des bibliothèques nécessaires pour interagir avec la base SQLite
from pathlib import Path
import sqlite3
import pandas as pd

# Chemin vers la base brute placée dans data/raw/
db_path = Path("data/raw/risk_monitor_dataset.sqlite")

# Vérification que le fichier existe avant d'ouvrir la connexion
assert db_path.exists(), f"Fichier introuvable : {db_path}"
                                                                                                                                                                              
# Ouverture de la connexion SQLite
conn = sqlite3.connect(db_path)                                                                          

## Inventaire des tables

On commence par identifier les tables présentes dans la base et récupérer leur définition SQL.  
Cela permet de vérifier le périmètre exact des données disponibles et de comprendre la structure générale de la base avant toute analyse métier.

In [9]:
# Requête système SQLite pour lister toutes les tables de la base
tables_df = pd.read_sql_query(
    "SELECT name, sql FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn
)

# Affichage du résultat pour vérifier le contenu de la base
tables_df

,name,sql
0,complaints,CREATE TABLE complaints (\n id INTE...
1,memberships,CREATE TABLE memberships (\n id INT...
2,payments,CREATE TABLE payments (\n id INTEGE...
3,subscriptions,CREATE TABLE subscriptions (\n id I...
4,users,CREATE TABLE users (\n id INTEGER P...


### Observation

La base contient cinq tables : `users`, `subscriptions`, `memberships`, `payments` et `complaints`.

À ce stade, leur rôle supposé est le suivant :
- `users` : informations sur les utilisateurs
- `subscriptions` : abonnements partagés et leurs caractéristiques
- `memberships` : appartenance d’un utilisateur à un abonnement
- `payments` : paiements liés aux abonnements
- `complaints` : réclamations ou tickets liés à un utilisateur ou un abonnement

Aucun dictionnaire de données n’est fourni. Les colonnes, les statuts et certaines relations métier devront donc être déduits à partir des données elles-mêmes.

## Schéma des tables

On inspecte maintenant la structure de chaque table afin d’identifier :
- les noms de colonnes,
- les types de données,
- les colonnes probablement clés,
- les colonnes de date,
- les colonnes de statut,
- les colonnes utiles pour les jointures entre tables.

In [10]:
# Inspection de la structure de chaque table pour comprendre les colonnes disponibles
for table in ["users", "subscriptions", "memberships", "payments", "complaints"]:
    print(f"\n--- Schéma de la table {table} ---")
    display(pd.read_sql_query(f"PRAGMA table_info({table});", conn))


--- Schéma de la table users ---


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,email,TEXT,0,None,0
2,2,country,TEXT,0,None,0
3,3,signup_date,TEXT,0,None,0
4,4,status,INTEGER,0,None,0
5,5,last_seen,TEXT,0,None,0
6,6,referral_code,TEXT,0,None,0
7,7,phone_prefix,TEXT,0,None,0



--- Schéma de la table subscriptions ---


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,brand,TEXT,0,None,0
2,2,owner_id,INTEGER,0,None,0
3,3,created_at,TEXT,0,None,0
4,4,status,INTEGER,0,None,0
5,5,max_slots,INTEGER,0,None,0
6,6,price_cents,INTEGER,0,None,0
7,7,currency,TEXT,0,None,0



--- Schéma de la table memberships ---


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,0
1,1,user_id,INTEGER,0,None,0
2,2,subscription_id,INTEGER,0,None,0
3,3,status,INTEGER,0,None,0
4,4,joined_at,TEXT,0,None,0
5,5,left_at,TEXT,0,None,0
6,6,reason,TEXT,0,None,0



--- Schéma de la table payments ---


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,user_id,INTEGER,0,None,0
2,2,subscription_id,INTEGER,0,None,0
3,3,amount_cents,INTEGER,0,None,0
4,4,fee_cents,INTEGER,0,None,0
5,5,status,TEXT,0,None,0
6,6,created_at,TEXT,0,None,0
7,7,captured_at,TEXT,0,None,0
8,8,currency,TEXT,0,None,0
9,9,stripe_error_code,TEXT,0,None,0



--- Schéma de la table complaints ---


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,reporter_id,INTEGER,0,None,0
2,2,target_id,INTEGER,0,None,0
3,3,subscription_id,INTEGER,0,None,0
4,4,type,TEXT,0,None,0
5,5,status,TEXT,0,None,0
6,6,created_at,TEXT,0,None,0
7,7,resolved_at,TEXT,0,None,0
8,8,resolution,TEXT,0,None,0


### Lecture du rôle des tables

À ce stade, la structure laisse penser que :

- **`users`** contient les informations de base sur chaque utilisateur : identité, pays, dates de présence, statut et éléments de contact ou de parrainage.
- **`subscriptions`** représente les abonnements partagés eux-mêmes : marque, propriétaire, date de création, capacité, prix et devise.
- **`memberships`** relie un utilisateur à un abonnement et semble décrire son appartenance dans le temps, avec un statut, une date d’entrée, une date de sortie et un motif éventuel.
- **`payments`** trace les paiements associés aux utilisateurs et aux abonnements, avec les montants, les frais, le statut de paiement et les horodatages.
- **`complaints`** recense les réclamations ou incidents déclarés par un utilisateur contre un autre utilisateur ou un abonnement, avec leur type, leur statut, leurs dates et leur résolution.

Un point notable est que la table **`memberships`** n’a pas de clé primaire explicitement déclarée dans le schéma affiché, contrairement aux autres tables. Cela devra être vérifié plus tard lors de l’exploration des doublons et des identifiants.

## Volumes par table

Le volume de chaque table permet de mesurer rapidement l’ampleur de la base et de vérifier si certaines tables sont particulièrement denses ou au contraire très petites, ce qui peut avoir un impact sur le scoring et sur l’interface.

In [11]:
# Comptage des lignes par table pour mesurer le volume de données disponible
for table in ["users", "subscriptions", "memberships", "payments", "complaints"]:
    count = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {table};", conn).iloc[0]["n"]
    print(f"{table}: {count}")

users: 2001
subscriptions: 400
memberships: 1083
payments: 7277
complaints: 1213


### Observation sur les volumes

La base contient :

- **2001 utilisateurs** dans `users`
- **400 abonnements** dans `subscriptions`
- **1083 memberships** dans `memberships`
- **7277 paiements** dans `payments`
- **1213 réclamations** dans `complaints`

Ces volumes suggèrent une base où :
- les utilisateurs sont plus nombreux que les abonnements, ce qui est cohérent avec un modèle de marketplace ;
- la table `payments` est la plus volumineuse, ce qui en fait une source centrale pour le scoring ;
- `complaints` représente une part significative des événements métier et devra être intégrée comme signal de risque ;
- `memberships` contient plus d’entrées que `subscriptions`, ce qui indique plusieurs membres par abonnement et confirme la logique de partage.

Ces ordres de grandeur seront utiles pour interpréter les anomalies, construire les features et choisir les agrégations pertinentes à l’étape de scoring.

## Aperçu rapide des valeurs réelles

On affiche maintenant quelques lignes de chaque table pour voir les formats réels des colonnes, les valeurs présentes, les éventuels champs manquants et les incohérences visibles dès la première lecture.

In [12]:
# Aperçu rapide des premières lignes de chaque table pour comprendre les valeurs réelles
for table in ["users", "subscriptions", "memberships", "payments", "complaints"]:
    print(f"\n--- Aperçu de {table} ---")
    display(pd.read_sql_query(f"SELECT * FROM {table} LIMIT 5;", conn))


--- Aperçu de users ---


,id,email,country,signup_date,status,last_seen,referral_code,phone_prefix
0,1,user_1@yahoo.fr,IT,2021-02-09 21:26:47,1,2021-04-28 14:23:00,NaN,+49
1,2,user_2@gmail.com,CH,2022-01-09T06:49:44Z,0,2023-01-05 17:03:20,NaN,+41
2,3,user_3@outlook.com,BE,1584765297,1,2021-07-30 05:38:37,27cb14dc,+32
3,4,user_4@hotmail.com,DE,2020-07-13T23:53:58Z,0,2024-06-10 09:06:04,NaN,+49
4,5,user_5@outlook.com,DE,21/07/2020 08:54,0,2026-03-06 20:16:50,NaN,+49



--- Aperçu de subscriptions ---


,id,brand,owner_id,created_at,status,max_slots,price_cents,currency
0,1,Microsoft 365,1,2023-01-14 23:23:57,0,4,1070,eur
1,2,HBO Max,1670,2023-07-16 21:39:51,2,4,233,USD
2,3,ChatGPT Plus,558,2024-10-02 13:54:25,0,3,1367,EUR
3,4,Microsoft 365,567,2023-10-26 23:03:50,0,6,460,EUR
4,5,NordVPN,950,2024-11-30 08:35:40,0,4,464,EUR



--- Aperçu de memberships ---


,id,user_id,subscription_id,status,joined_at,left_at,reason
0,563,1485,231,1,2024-12-06 21:23:01,NaN,NaN
1,1042,690,399,2,2024-09-23 02:25:46,2025-03-31 22:56:44,fraud
2,72,56,31,1,2025-02-15 18:19:15,NaN,NaN
3,414,590,168,1,2024-08-06 15:33:32,NaN,NaN
4,413,20,168,1,2024-05-19 10:34:05,NaN,NaN



--- Aperçu de payments ---


,id,user_id,subscription_id,amount_cents,fee_cents,status,created_at,captured_at,currency,stripe_error_code
0,1,1485,231,1322,78,succeeded,2024-12-04T21:23:01+02:00,2024-12-07 14:23:01,EUR,None
1,2,1485,231,366,37,succeeded,2025-01-03T21:23:01+01:00,2025-01-05 11:23:01,eur,None
2,3,1485,231,238,28,succeeded,2025-02-07 21:23:01,2025-02-10 04:23:01,EUR,None
3,4,1485,231,212,23,succeeded,2025-03-04 21:23:01,2025-03-06 20:23:01,EUR,None
4,5,1485,231,1325,144,succeeded,2025-04-03 21:23:01,2025-04-04 23:23:01,EUR,None



--- Aperçu de complaints ---


,id,reporter_id,target_id,subscription_id,type,status,created_at,resolved_at,resolution
0,1,904,1565,389,billing_issue,open,2023-09-06 08:02:33,NaN,NaN
1,2,1702,600,260,subscription_inactive,escalated,2024-07-30 12:44:00,NaN,NaN
2,3,1361,672,124,Accès refusé,resolved,2022-10-01 21:19:13,2024-01-27 20:51:20,
3,4,19,1706,317,ACCESS_DENIED,RESOLVED,2024-05-08 04:05:41,2024-06-03 03:41:59,refunded
4,5,327,609,289,ACCESS_DENIED,closed,2021-06-11 13:01:44,2021-10-31 06:56:01,refunded


### Premières observations sur les données

L’aperçu des premières lignes met déjà en évidence plusieurs points importants :

- La colonne `signup_date` dans `users` contient des formats de dates hétérogènes :
  - format texte classique (`2021-02-09 21:26:47`)
  - format ISO avec timezone (`2022-01-09T06:49:44Z`)
  - timestamp numérique (`1584765297`)
  - format français (`21/07/2020 08:54`)
  
  Cette diversité confirme qu’une normalisation des dates sera nécessaire.

- La colonne `status` de `users` semble contenir des valeurs codées numériquement, mais le sens exact devra être déduit plus tard à partir du contexte.

- Dans `subscriptions`, les données semblent globalement cohérentes au premier regard, mais on observe déjà une variation de casse sur `currency` :
  - `eur`
  - `USD`
  - `EUR`
  
  Cela devra être harmonisé.

- Dans `memberships`, la colonne `status` contient des valeurs numériques (`1`, `2`) alors que les colonnes `id`, `user_id` et `subscription_id` suggèrent une table de liaison. Le champ `reason` peut servir d’indicateur métier utile, par exemple `fraud`.

- Dans `payments`, les horodatages sont encore une fois hétérogènes :
  - `created_at` peut contenir des timestamps avec timezone (`2024-12-04T21:23:01+02:00`)
  - `captured_at` apparaît sans timezone
  - `currency` varie en casse
  
  La colonne `status` est textuelle ici, ce qui diffère des tables `users` et `memberships`.

- Dans `complaints`, les valeurs de `type` et `status` montrent déjà des variantes de casse et de langue :
  - `billing_issue`
  - `subscription_inactive`
  - `Accès refusé`
  - `ACCESS_DENIED`
  
  Cela confirme qu’un travail de normalisation lexicale sera nécessaire.

Ces premiers exemples montrent que la base est bien volontairement dégradée et qu’il faudra à la fois :
- normaliser les formats,
- harmoniser les statuts,
- identifier les doublons,
- et documenter toutes les hypothèses retenues.

## Exploration des statuts

On examine maintenant plus en détail les valeurs de `status` pour chaque table.  
L’objectif est de repérer :
- les codes numériques non documentés,
- les statuts textuels incohérents,
- les variantes de casse,
- les valeurs manquantes ou suspectes.

Cette étape prépare le reverse-engineering des statuts à l’étape suivante.

In [13]:
# Analyse des valeurs de status pour identifier les codes, variantes de casse et valeurs suspectes
for table in ["users", "subscriptions", "memberships", "payments", "complaints"]:
    print(f"\n--- Status observés dans {table} ---")
    display(
        pd.read_sql_query(
            f"""
            SELECT status, COUNT(*) AS n
            FROM {table}
            GROUP BY status
            ORDER BY n DESC;
            """,
            conn
        )
    )


--- Status observés dans users ---


,status,n
0,0.0,1325
1,4.0,198
2,1.0,179
3,3.0,110
4,2.0,107
5,-1.0,38
6,NaN,25
7,99.0,19



--- Status observés dans subscriptions ---


,status,n
0,0,229
1,2,68
2,3,57
3,1,46



--- Status observés dans memberships ---


,status,n
0,1,428
1,2,218
2,3,107
3,5,84
4,6,68
5,4,64
6,0,59
7,7,55



--- Status observés dans payments ---


,status,n
0,succeeded,3322
1,failed,1774
2,pending,581
3,refunded,375
4,Succeeded,232
5,success,216
6,disputed,216
7,suceeded,210
8,FAILED,210
9,canceled,141



--- Status observés dans complaints ---


,status,n
0,in_progress,194
1,open,192
2,RESOLVED,171
3,resolved,169
4,escalated,165
5,Open,164
6,closed,158


### Observation sur les statuts observés

L’exploration des statuts confirme que la base contient plusieurs conventions de codage différentes selon les tables.

#### `users`
La table `users` contient des statuts numériques avec des valeurs qui semblent représenter plusieurs états métier, mais dont la signification exacte reste à déduire :
- `0`
- `1`
- `2`
- `3`
- `4`

On observe aussi des valeurs atypiques :
- `-1`
- `99`
- `NaN`

Ces valeurs devront être traitées comme des cas particuliers, car elles peuvent correspondre à des états invalides, inconnus ou exceptionnels.

#### `subscriptions`
La table `subscriptions` contient aussi des statuts numériques, mais avec une palette plus réduite :
- `0`
- `1`
- `2`
- `3`

Cela suggère un cycle de vie de l’abonnement plus simple que celui des utilisateurs.

#### `memberships`
La table `memberships` contient des statuts numériques répartis de `0` à `7`.

Le fait que les codes soient nombreux et non documentés confirme qu’il faudra les reverse-engineer à partir du contexte métier et des autres champs de la table, en particulier `joined_at`, `left_at` et `reason`.

#### `payments`
La table `payments` utilise des statuts textuels, mais avec plusieurs variantes pour un même concept :
- `succeeded`
- `Succeeded`
- `success`
- `suceeded`
- `failed`
- `FAILED`
- `pending`
- `refunded`
- `disputed`
- `canceled`

On observe donc :
- des variations de casse,
- une faute d’orthographe probable avec `suceeded`,
- plusieurs termes proches qui devront être regroupés.

#### `complaints`
La table `complaints` contient également des statuts textuels avec variantes de casse :
- `open`
- `Open`
- `in_progress`
- `escalated`
- `resolved`
- `RESOLVED`
- `closed`

Ces statuts devront être normalisés pour rendre les analyses et agrégations fiables.

#### Conclusion provisoire
Les statuts ne sont pas homogènes d’une table à l’autre :
- certaines tables utilisent des **codes numériques**,
- d’autres utilisent des **libellés textuels**,
- certains libellés existent en plusieurs variantes,
- certaines valeurs semblent invalides ou rares.

Cette observation confirme qu’une phase de normalisation et de reverse-engineering sera nécessaire avant de construire le scoring.

## Recherche de doublons et de répétitions métier

Le schéma montre que certaines tables peuvent contenir plusieurs enregistrements pour un même événement métier, et que la table `memberships` n’a pas de clé primaire déclarée dans le schéma affiché.

On cherche ici :
- des doublons stricts sur des combinaisons de colonnes métier,
- des répétitions d’un même événement avec statuts différents,
- des cas qui devront être distingués entre vrai doublon, historique légitime et anomalie.

In [14]:
# Recherche de doublons sur les principales tables à partir de colonnes métier,
# afin de distinguer un vrai doublon d'un simple identifiant différent.

duplicate_queries = {
    "users": """
        SELECT
            email, country, signup_date, status, last_seen, referral_code, phone_prefix,
            COUNT(*) AS n
        FROM users
        GROUP BY email, country, signup_date, status, last_seen, referral_code, phone_prefix
        HAVING COUNT(*) > 1
        ORDER BY n DESC;
    """,
    "subscriptions": """
        SELECT
            brand, owner_id, created_at, status, max_slots, price_cents, currency,
            COUNT(*) AS n
        FROM subscriptions
        GROUP BY brand, owner_id, created_at, status, max_slots, price_cents, currency
        HAVING COUNT(*) > 1
        ORDER BY n DESC;
    """,
    "memberships": """
        SELECT
            user_id, subscription_id, status, joined_at, left_at, reason,
            COUNT(*) AS n
        FROM memberships
        GROUP BY user_id, subscription_id, status, joined_at, left_at, reason
        HAVING COUNT(*) > 1
        ORDER BY n DESC;
    """,
    "payments": """
        SELECT
            user_id, subscription_id, amount_cents, fee_cents, status,
            created_at, captured_at, currency, stripe_error_code,
            COUNT(*) AS n
        FROM payments
        GROUP BY user_id, subscription_id, amount_cents, fee_cents, status,
                 created_at, captured_at, currency, stripe_error_code
        HAVING COUNT(*) > 1
        ORDER BY n DESC;
    """,
    "complaints": """
        SELECT
            reporter_id, target_id, subscription_id, type, status,
            created_at, resolved_at, resolution,
            COUNT(*) AS n
        FROM complaints
        GROUP BY reporter_id, target_id, subscription_id, type, status,
                 created_at, resolved_at, resolution
        HAVING COUNT(*) > 1
        ORDER BY n DESC;
    """
}

# Exécution de chaque requête pour visualiser les doublons éventuels
for table, query in duplicate_queries.items():
    print(f"\n--- Doublons détectés dans {table} ---")
    df = pd.read_sql_query(query, conn)
    display(df)


--- Doublons détectés dans users ---


,email,country,signup_date,status,last_seen,referral_code,phone_prefix,n



--- Doublons détectés dans subscriptions ---


,brand,owner_id,created_at,status,max_slots,price_cents,currency,n



--- Doublons détectés dans memberships ---


,user_id,subscription_id,status,joined_at,left_at,reason,n



--- Doublons détectés dans payments ---


,user_id,subscription_id,amount_cents,fee_cents,status,created_at,captured_at,currency,stripe_error_code,n
0,720,398,657,51,failed,2022-02-03 03:06:10,NaN,EUR,stolen_card,3
1,79,379,224,25,succeeded,2025-05-07 00:25:02,2025-05-07 16:25:02,EUR,NaN,2
2,79,379,828,66,pending,2025-04-05 00:25:02,NaN,EUR,NaN,2
3,458,193,1046,141,refunded,2024-07-10 22:59:25,NaN,EUR,NaN,2
4,574,145,768,64,succeeded,2022-09-23 04:10:39,2022-09-25 21:10:39,eur,NaN,2
5,590,168,727,88,pending,2024-10-06 15:33:32,NaN,EUR,NaN,2
6,601,289,323,37,failed,2023-02-14 17:48:03,NaN,EUR,stolen_card,2
7,720,398,366,42,succeeded,2022-05-02T03:06:10+02:00,2022-05-02 07:06:10,EUR,NaN,2
8,771,63,237,19,failed,2025-05-05 12:00:38,NaN,USD,incorrect_cvc,2
9,884,300,1229,171,succeeded,2025-03-28 21:16:25,2025-03-29 07:16:25,EUR,NaN,2



--- Doublons détectés dans complaints ---


,reporter_id,target_id,subscription_id,type,status,created_at,resolved_at,resolution,n


### Lecture des résultats sur les doublons

Les requêtes n’ont pas mis en évidence de doublons stricts dans les tables `users`, `subscriptions`, `memberships` et `complaints` sur les colonnes métier utilisées pour le regroupement.

En revanche, la table `payments` contient plusieurs combinaisons répétées exactement deux fois, et un cas à trois occurrences. Cela indique qu’il existe des paiements strictement identiques sur les colonnes observées, ou des événements très proches ayant été enregistrés plusieurs fois de manière indiscernable avec cette méthode.

Quelques cas ressortent particulièrement :
- des doublons avec `status = failed`, souvent associés à un `stripe_error_code` comme `stolen_card`, `card_declined`, `fraudulent` ou `do_not_honor`,
- des doublons avec `status = succeeded`,
- des doublons avec `status = pending`,
- un cas avec `status = refunded`,
- un cas avec `status = canceled`.

Ce résultat confirme que la table `payments` devra être traitée avec prudence à l’étape de nettoyage :
- il faudra distinguer les vrais doublons des répétitions légitimes,
- il faudra vérifier si certains doublons correspondent à des tentatives de paiement répétées,
- il faudra décider si la suppression, le regroupement ou le flag est la meilleure approche selon le contexte.

Les doublons partiels restent encore à explorer, mais cette première analyse montre déjà que `payments` est la table la plus sensible sur ce point.

## Vérification des relations entre les tables

Avant de nettoyer, on teste la cohérence des relations probables entre les tables pour repérer les clés orphelines.

L’objectif est de vérifier quelles jointures seront fiables pour la suite, en particulier pour :
- le scoring,
- l’historique détaillé d’un subscriber,
- et la consolidation des signaux de risque.

Certaines relations, comme `complaints.target_id -> users.id`, restent une hypothèse métier à confirmer par l’exploration.

In [15]:
# Vérification de la cohérence des relations entre tables.
# Les relations testées ici sont les plus probables d'après le contexte métier.
# Le champ target_id dans complaints est testé comme s'il pointait vers users.id,
# mais cette hypothèse devra être confirmée ou infirmée plus tard.

orphan_checks = {
    "subscriptions.owner_id -> users.id": """
        SELECT COUNT(*) AS orphan_count
        FROM subscriptions s
        LEFT JOIN users u ON u.id = s.owner_id
        WHERE s.owner_id IS NOT NULL AND u.id IS NULL;
    """,
    "memberships.user_id -> users.id": """
        SELECT COUNT(*) AS orphan_count
        FROM memberships m
        LEFT JOIN users u ON u.id = m.user_id
        WHERE m.user_id IS NOT NULL AND u.id IS NULL;
    """,
    "memberships.subscription_id -> subscriptions.id": """
        SELECT COUNT(*) AS orphan_count
        FROM memberships m
        LEFT JOIN subscriptions s ON s.id = m.subscription_id
        WHERE m.subscription_id IS NOT NULL AND s.id IS NULL;
    """,
    "payments.user_id -> users.id": """
        SELECT COUNT(*) AS orphan_count
        FROM payments p
        LEFT JOIN users u ON u.id = p.user_id
        WHERE p.user_id IS NOT NULL AND u.id IS NULL;
    """,
    "payments.subscription_id -> subscriptions.id": """
        SELECT COUNT(*) AS orphan_count
        FROM payments p
        LEFT JOIN subscriptions s ON s.id = p.subscription_id
        WHERE p.subscription_id IS NOT NULL AND s.id IS NULL;
    """,
    "complaints.reporter_id -> users.id": """
        SELECT COUNT(*) AS orphan_count
        FROM complaints c
        LEFT JOIN users u ON u.id = c.reporter_id
        WHERE c.reporter_id IS NOT NULL AND u.id IS NULL;
    """,
    "complaints.target_id -> users.id (hypothèse)": """
        SELECT COUNT(*) AS orphan_count
        FROM complaints c
        LEFT JOIN users u ON u.id = c.target_id
        WHERE c.target_id IS NOT NULL AND u.id IS NULL;
    """,
    "complaints.subscription_id -> subscriptions.id": """
        SELECT COUNT(*) AS orphan_count
        FROM complaints c
        LEFT JOIN subscriptions s ON s.id = c.subscription_id
        WHERE c.subscription_id IS NOT NULL AND s.id IS NULL;
    """
}

# Exécution de chaque test de cohérence pour repérer les lignes orphelines
for relation, query in orphan_checks.items():
    orphan_count = pd.read_sql_query(query, conn).iloc[0]["orphan_count"]
    print(f"{relation}: {orphan_count}")

subscriptions.owner_id -> users.id: 10
memberships.user_id -> users.id: 0
memberships.subscription_id -> subscriptions.id: 15
payments.user_id -> users.id: 30
payments.subscription_id -> subscriptions.id: 121
complaints.reporter_id -> users.id: 0
complaints.target_id -> users.id (hypothèse): 0
complaints.subscription_id -> subscriptions.id: 20


### Lecture des relations entre tables

La vérification des relations montre que plusieurs jointures sont globalement fiables, mais qu’il existe aussi quelques lignes orphelines.

#### Relations cohérentes
- `memberships.user_id -> users.id` : **0** ligne orpheline
- `complaints.reporter_id -> users.id` : **0** ligne orpheline
- `complaints.target_id -> users.id` (hypothèse) : **0** ligne orpheline

Ces relations semblent donc exploitables sans correction particulière à ce stade.

#### Relations avec anomalies
- `subscriptions.owner_id -> users.id` : **10** lignes orphelines
- `memberships.subscription_id -> subscriptions.id` : **15** lignes orphelines
- `payments.user_id -> users.id` : **30** lignes orphelines
- `payments.subscription_id -> subscriptions.id` : **121** lignes orphelines
- `complaints.subscription_id -> subscriptions.id` : **20** lignes orphelines

Ces écarts indiquent soit :
- des données incomplètes,
- des suppressions ou références cassées,
- des erreurs d’injection,
- ou des lignes volontairement dégradées dans le dataset.

#### Conséquence pour la suite
Les jointures avec `users` semblent plus fiables que celles avec `subscriptions` dans certaines tables, mais rien ne doit encore être supprimé à ce stade.

Ces anomalies devront être prises en compte à l’étape de nettoyage :
- certaines lignes pourront être conservées avec un flag d’anomalie,
- d’autres devront peut-être être exclues des agrégations selon leur impact sur le scoring.

Cette vérification confirme que la base est partiellement incohérente et qu’il faudra documenter précisément les choix de traitement.

## Inspection des colonnes de dates

Le test précise que les timestamps ne sont pas homogènes : certains sont textuels, d’autres au format ISO, d’autres encore sous forme de timestamp numérique.

On observe ici les valeurs brutes des colonnes de dates avant toute normalisation, afin de préparer un nettoyage cohérent à l’étape suivante.

In [16]:
# Inspection brute des colonnes de dates pour repérer les formats hétérogènes
# avant toute tentative de normalisation.

date_columns_checks = {
    "users": ["signup_date", "last_seen"],
    "subscriptions": ["created_at"],
    "memberships": ["joined_at", "left_at"],
    "payments": ["created_at", "captured_at"],
    "complaints": ["created_at", "resolved_at"]
}

for table, cols in date_columns_checks.items():
    print(f"\n--- Dates dans {table} ---")
    for col in cols:
        print(f"\nColonne: {col}")
        display(
            pd.read_sql_query(
                f"""
                SELECT {col}, COUNT(*) AS n
                FROM {table}
                WHERE {col} IS NOT NULL
                GROUP BY {col}
                ORDER BY n DESC
                LIMIT 10;
                """,
                conn
            )
        )


--- Dates dans users ---

Colonne: signup_date


,signup_date,n
0,31/08/2024 10:41,1
1,31/07/2024 22:16,1
2,31/07/2024 15:45,1
3,31/05/2020 19:47,1
4,31/03/2024 22:53,1
5,31/03/2021 13:55,1
6,31/03/2021 05:35,1
7,30/11/2023 04:04,1
8,30/07/2024 04:56,1
9,30/07/2022 08:42,1



Colonne: last_seen


,last_seen,n
0,2026-04-10 02:33:54,1
1,2026-04-09 18:25:45,1
2,2026-04-09 12:13:51,1
3,2026-04-09 10:48:23,1
4,2026-04-09 09:09:12,1
5,2026-04-09 05:13:59,1
6,2026-04-08 06:34:05,1
7,2026-04-07 07:56:02,1
8,2026-04-06 22:51:14,1
9,2026-04-06 19:56:47,1



--- Dates dans subscriptions ---

Colonne: created_at


...

,captured_at,n
0,2024-06-15 01:00:00,20
1,2024-11-29 10:01:00,10
2,2025-05-07 16:25:02,2
3,2025-03-29 07:16:25,2
4,2025-03-26 17:36:15,2
5,2024-09-17 18:58:33,2
6,2024-05-26 19:11:18,2
7,2024-04-26 01:14:55,2
8,2022-09-25 21:10:39,2
9,2022-08-11 22:04:36,2



--- Dates dans complaints ---

Colonne: created_at


,created_at,n
0,2025-05-11 21:55:02,1
1,2025-05-11 15:33:39,1
2,2025-05-10 05:15:49,1
3,2025-05-08 19:06:41,1
4,2025-05-08 06:56:26,1
5,2025-05-07 18:00:04,1
6,2025-05-06 06:23:38,1
7,2025-05-05 23:42:19,1
8,2025-05-05 06:55:57,1
9,2025-05-04 05:31:11,1



Colonne: resolved_at


,resolved_at,n
0,2025-05-30 23:30:17,1
1,2025-05-28 06:47:33,1
2,2025-05-27 10:31:55,1
3,2025-05-27 01:33:44,1
4,2025-05-26 07:21:36,1
5,2025-05-26 07:20:36,1
6,2025-05-25 16:56:13,1
7,2025-05-25 15:14:34,1
8,2025-05-25 01:56:49,1
9,2025-05-25 00:09:13,1


### Lecture des colonnes de dates

L’inspection des dates confirme que la base contient plusieurs formats et quelques particularités importantes.

#### `users`
- `signup_date` est hétérogène : certaines valeurs sont au format texte français (`31/08/2024 10:41`), d’autres au format ISO, et d’autres encore sous forme de timestamp numérique dans les premières observations de la table.
- `last_seen` est plus homogène et semble principalement au format `YYYY-MM-DD HH:MM:SS`.

Cela confirme qu’une normalisation robuste des dates sera nécessaire, en particulier pour `signup_date`.

#### `subscriptions`
- `created_at` semble globalement homogène dans l’extrait observé, au format `YYYY-MM-DD HH:MM:SS`.

#### `memberships`
- `joined_at` et `left_at` sont également au format `YYYY-MM-DD HH:MM:SS` dans l’échantillon observé.
- Il faudra néanmoins vérifier plus tard si des valeurs manquantes ou incohérentes existent sur l’ensemble de la table, notamment pour `left_at`.

#### `payments`
- `created_at` présente plusieurs répétitions identiques, avec des volumes notables sur certaines dates, par exemple `2024-06-15 00:00:00`.
- `captured_at` suit globalement le même format, mais on observe aussi des répétitions qui suggèrent des événements récurrents ou des regroupements de données.
- Dans l’aperçu précédent, certaines valeurs de `created_at` incluaient des fuseaux horaires, ce qui confirme que le traitement des dates devra gérer plusieurs formats.

#### `complaints`
- `created_at` et `resolved_at` semblent cohérents dans l’échantillon observé, au format `YYYY-MM-DD HH:MM:SS`.
- La présence de `resolved_at` permet de calculer des délais de traitement ou de résolution, ce qui pourra être utile plus tard pour l’analyse métier.

#### Conclusion provisoire
Les dates sont globalement exploitables, mais elles ne sont pas toutes homogènes :
- certains champs mélangent plusieurs formats,
- certains timestamps sont avec fuseau horaire, d’autres non,
- certaines dates peuvent être répétées massivement,
- certaines colonnes pourront être exploitées pour des durées ou des récences.

La prochaine étape devra donc :
1. normaliser les dates,
2. documenter les règles de conversion,
3. vérifier les valeurs impossibles ou incohérentes,
4. préparer les champs temporels pour le scoring.

## Synthèse de l’étape 1

À ce stade, la structure de la base est mieux comprise :
- les tables principales sont identifiées,
- les volumes sont connus,
- les statuts sont observés,
- les doublons potentiels sont repérés,
- les relations entre tables sont testées,
- les formats de dates sont inventoriés.

Cette lecture servira de base à l’étape 2 : nettoyage documenté, normalisation des statuts et traitement des anomalies.